# Entrenamiento de la tesis en Colab (v3: con las fotos de ODK)

1. `tesis_colab.zip` debe estar en tu Drive, en **`MyDrive/tesis`**.
2. *Entorno de ejecución → Cambiar tipo de entorno → GPU A100*.
3. *Entorno de ejecución → Ejecutar todas*.

Corre los 7 entrenamientos del protocolo, **3 a la vez** en la misma GPU, cuantiza cada uno a int8 y lo evalúa. La celda de avance se refresca cada minuto. Cada run terminado queda en `MyDrive/tesis/runs_odk/` (los runs sin ODK quedan en `runs/`). El experimento B usa solo el histórico, que no cambió: sus runs se copian de `runs/` en lugar de reentrenarse. Si Colab se desconecta, *Ejecutar todas* de nuevo: lo terminado se salta y un run ya entrenado retoma desde la cuantización.

Al final, el **experimento de cuantización** compara cuatro formatos para el teléfono sobre los modelos ya entrenados, y el
**análisis según la planificación** mide la verificación contra convenio, la hipótesis 5.8 y los errores por cadena
(ninguno de los dos reentrena). Todas las métricas se calculan sobre la prueba sin duplicados.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/tesis'
RUNS_DIR = f'{DRIVE}/runs_odk'  # v3: con las fotos de ODK; los runs anteriores quedan en runs/
!mkdir -p "{RUNS_DIR}"
import glob, os, shutil
for r in glob.glob(f'{DRIVE}/runs/B_*'):  # el experimento B usa solo el histórico: no cambia con ODK
    if not os.path.exists(f'{RUNS_DIR}/{os.path.basename(r)}'):
        shutil.copytree(r, f'{RUNS_DIR}/{os.path.basename(r)}')
!nvidia-smi -L

In [ ]:
# Copia al disco local de Colab: leer 58 000 fotos desde Drive en cada época sería lentísimo.
%cd /content
!cp "{DRIVE}/tesis_colab.zip" /content/ && unzip -q -o tesis_colab.zip -d /content/tesis && rm tesis_colab.zip
%cd /content/tesis

Código actualizado respecto del zip: fotos decodificadas en RAM tras la primera época y aumentos en la GPU (mismo protocolo, más rápido), y las métricas del análisis (el bootstrap sobre un subconjunto de clases y el convenio aproximado de cada local, `metrics.contracts`). Van antes de cualquier `import tesis...`: Python guarda el módulo importado y no vería la versión nueva.

In [ ]:
%%writefile tesis/data.py
"""Carga de imágenes con tf.data. El enmascaramiento de la superposición vive dentro del modelo
(ver train.build_model), así la versión TFLite lo aplica igual en el dispositivo."""
import keras
import tensorflow as tf

from .config import CLASSES, IMG_SIZE, SEED

AUGMENT = keras.Sequential([
    keras.layers.RandomRotation(0.03, fill_mode="constant"),
    keras.layers.RandomZoom((-0.15, 0.0)),  # recorte aleatorio
    keras.layers.RandomTranslation(0.05, 0.05, fill_mode="constant"),
    keras.layers.RandomBrightness(0.25, value_range=(0, 255)),
    keras.layers.RandomContrast(0.25),
])
# ponytail: sin capa de perspectiva nativa en Keras; RandomZoom+Translation la aproximan.


def _load(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(tf.clip_by_value(tf.round(img), 0, 255), tf.uint8), label  # uint8: 4x menos RAM en caché


def _jpeg(img, label):
    img = tf.image.random_jpeg_quality(img, 40, 95)  # compresión con pérdida
    return tf.cast(img, tf.float32), label


def _float(img, label):
    return tf.cast(img, tf.float32), label


def dataset(df, batch=32, train=False):
    """df con columnas path y label (nombre de clase). Imágenes float32 en [0, 255].
    Las fotos se decodifican una vez y quedan en RAM (cache); los aumentos se sortean en cada época."""
    labels = df.label.map(CLASSES.index).values
    ds = tf.data.Dataset.from_tensor_slices((df.path.values, labels))
    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE).cache()
    if train:  # AUGMENT no va aquí sino en train.py, envolviendo al modelo: así corre en la GPU
        ds = ds.shuffle(len(df), seed=SEED).map(_jpeg, num_parallel_calls=tf.data.AUTOTUNE).batch(batch)
    else:
        ds = ds.map(_float, num_parallel_calls=tf.data.AUTOTUNE).batch(batch)
    return ds.prefetch(tf.data.AUTOTUNE)


In [ ]:
%%writefile tesis/train.py
"""Entrenamiento por transferencia con protocolo común (sección 5.5).

Uso:
    python -m tesis.train --model mobilenetv3 --exp A
    python -m tesis.train --model resnet50 --exp A --no-mask   # control del atajo de la superposición
"""
import argparse
import json
from pathlib import Path

import keras
import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

from .config import CLASSES, IMG_SIZE, MASK_H, MASK_W, MODELS, SEED
from .data import AUGMENT, dataset
from .split import experiment


def overlay_mask():
    m = np.ones((IMG_SIZE, IMG_SIZE, 1), dtype="float32")
    m[-round(IMG_SIZE * MASK_H):, -round(IMG_SIZE * MASK_W):] = 0
    return m


def build_model(name, mask=True):
    """Entrada: RGB float32 [0, 255] de IMG_SIZE². Máscara y preprocesamiento incluidos en el grafo."""
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = inp * overlay_mask() if mask else inp
    shape = (IMG_SIZE, IMG_SIZE, 3)
    if name == "resnet50":
        x = keras.applications.resnet50.preprocess_input(x)
        base = keras.applications.ResNet50(include_top=False, weights="imagenet", input_shape=shape, pooling="avg")
    elif name == "mobilenetv3":
        base = keras.applications.MobileNetV3Large(include_top=False, weights="imagenet", input_shape=shape, pooling="avg")
    elif name == "efficientnetb0":
        base = keras.applications.EfficientNetB0(include_top=False, weights="imagenet", input_shape=shape, pooling="avg")
    else:
        raise ValueError(f"modelo desconocido: {name}")
    x = keras.layers.Dropout(0.2)(base(x))
    return keras.Model(inp, keras.layers.Dense(len(CLASSES), activation="softmax")(x), name=name)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", choices=MODELS, required=True)
    ap.add_argument("--exp", choices=["A", "B"], default="A")
    ap.add_argument("--no-mask", action="store_true")
    ap.add_argument("--manifest", default="data/manifest.csv")
    ap.add_argument("--stores", default="data/stores_split.csv")
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--batch", type=int, default=32)
    args = ap.parse_args()

    keras.utils.set_random_seed(SEED)
    df = pd.read_csv(args.manifest)
    parts = experiment(df[df.label.notna()], pd.read_csv(args.stores), args.exp)
    run = Path("runs") / f"{args.exp}_{args.model}{'_nomask' if args.no_mask else ''}"
    run.mkdir(parents=True, exist_ok=True)

    y = parts["train"].label.map(CLASSES.index).values
    present = np.unique(y)
    weights = dict(zip(present.tolist(), compute_class_weight("balanced", classes=present, y=y)))

    model = build_model(args.model, mask=not args.no_mask)
    # Los aumentos envuelven al modelo solo para entrenar (corren en la GPU); se guarda `model`, sin ellos.
    trainer = keras.Sequential([keras.Input((IMG_SIZE, IMG_SIZE, 3)), AUGMENT, model])
    trainer.compile(keras.optimizers.Adam(args.lr), "sparse_categorical_crossentropy", metrics=["accuracy"])
    trainer.fit(
        dataset(parts["train"], args.batch, train=True),
        validation_data=dataset(parts["val"], args.batch),
        epochs=args.epochs,
        class_weight=weights,
        callbacks=[
            keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.3),
            keras.callbacks.CSVLogger(run / "history.csv"),
        ],
    )
    model.save(run / "model.keras")
    for k, v in parts.items():
        v.to_csv(run / f"{k}.csv", index=False)
    json.dump(vars(args), open(run / "args.json", "w"), indent=2)
    print(f"guardado en {run}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile tesis/evaluate.py
"""Evalúa un run sobre su conjunto de prueba (Keras o TFLite) y escribe métricas en JSON.

Uso:
    python -m tesis.evaluate runs/A_mobilenetv3 model_int8.tflite --contracts data/contracts.csv
contracts.csv (opcional): store, label   -> una fila por convenio vigente del local.
"""
import argparse
import json
from pathlib import Path

import keras
import numpy as np
import pandas as pd
import tensorflow as tf

from .config import CLASSES
from .data import dataset
from .metrics import report, verification


def predict(model_path, df):
    ds = dataset(df, batch=1 if model_path.suffix == ".tflite" else 64)
    if model_path.suffix != ".tflite":
        return keras.models.load_model(model_path).predict(ds)
    try:
        it = tf.lite.Interpreter(str(model_path))
        it.allocate_tensors()
    except RuntimeError:  # XNNPACK no prepara el MobileNetV3 int8: kernels estándar, mismos resultados
        it = tf.lite.Interpreter(str(model_path), experimental_op_resolver_type=(
            tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES))
        it.allocate_tensors()
    inp, out = it.get_input_details()[0]["index"], it.get_output_details()[0]["index"]
    probs = []
    for x, _ in ds:
        it.set_tensor(inp, x.numpy())
        it.invoke()
        probs.append(it.get_tensor(out)[0])
    return np.array(probs)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("run")
    ap.add_argument("model", nargs="?", default="model.keras")
    ap.add_argument("--contracts")
    args = ap.parse_args()

    run = Path(args.run)
    test = pd.read_csv(run / "test.csv")
    probs = predict(run / args.model, test)
    y, stores = test.label.map(CLASSES.index).values, test.store.values

    contracts = None
    if args.contracts:
        contracts = pd.read_csv(args.contracts).groupby("store").label.agg(set).to_dict()
    m = report(y, probs, stores, contracts)
    if contracts:
        m.update(verification(probs, y, stores, contracts))
    m["size_mb"] = (run / args.model).stat().st_size / 2**20

    stem = Path(args.model).stem
    np.save(run / f"probs_{stem}.npy", probs)  # para bootstrap entre configuraciones
    json.dump(m, open(run / f"metrics_{stem}.json", "w"), indent=2, default=float)
    print(json.dumps({k: m[k] for k in ("f1_macro", "recall_macro", "reject_recall", "size_mb")}, indent=2, default=float))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile tesis/metrics.py
"""Métricas predictivas, verificación contra convenio y viabilidad (secciones 5.6–5.9).

`probs`: matriz (n, 7) en el orden de config.CLASSES. `y`: índices de clase verdadera.
`contracts`: dict store -> set de nombres de clase con convenio vigente.
"""
import numpy as np
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support, recall_score

from .config import CLASSES, REJECT, THRESHOLDS

IDX = {c: i for i, c in enumerate(CLASSES)}
REJECT_IDX = [IDX[c] for c in REJECT]
MUEBLES = [c for c in CLASSES if c not in REJECT]


def contracts(manifest):
    """Muebles con convenio por local, a falta de la maestra de convenios: los que aparecen en las fotos verificadas
    del local y los que ODK declara para él (el formulario solo despliega los contratados). Es un superconjunto del
    convenio: restringe la predicción menos que la maestra, no más."""
    out = {}
    for s, label, declared in manifest[manifest.verified == 1][["store", "label", "declarado"]].itertuples(index=False):
        for c in {label, declared} & set(MUEBLES):
            out.setdefault(s, set()).add(c)
    return out


def conditional_pred(probs, stores, contracts):
    """Argmax restringido a los muebles del local más las dos categorías de rechazo."""
    mask = np.zeros_like(probs, dtype=bool)
    for i, s in enumerate(stores):
        mask[i, [IDX[c] for c in contracts.get(s, CLASSES)] + REJECT_IDX] = True
    return np.where(mask, probs, -np.inf).argmax(1)


def report(y, probs, stores=None, contracts=None):
    pred = probs.argmax(1)
    labels = list(range(len(CLASSES)))
    p, r, _, n = precision_recall_fscore_support(y, pred, labels=labels, zero_division=0)
    out = {
        "f1_macro": f1_score(y, pred, labels=labels, average="macro", zero_division=0),
        "recall_macro": recall_score(y, pred, labels=labels, average="macro", zero_division=0),
        "per_class": {c: {"precision": p[i], "recall": r[i], "support": int(n[i])} for i, c in enumerate(CLASSES)},
        "reject_recall": min(r[i] for i in REJECT_IDX),
        "confusion": confusion_matrix(y, pred, labels=labels, normalize="true").round(3).tolist(),
    }
    if contracts is not None:
        out["conditional_accuracy"] = float((conditional_pred(probs, stores, contracts) == y).mean())
    return out


def bootstrap_f1_diff(y, pred_a, pred_b, stores, n=2000, seed=0, labels=None):
    """IC 95 % de F1-macro(a) - F1-macro(b) con remuestreo por local. `labels`: clases que promedia (todas)."""
    rng = np.random.default_rng(seed)
    stores = np.asarray(stores)
    groups = [np.flatnonzero(stores == s) for s in np.unique(stores)]
    labels = list(range(len(CLASSES))) if labels is None else labels
    diffs = []
    for _ in range(n):
        idx = np.concatenate([groups[i] for i in rng.integers(len(groups), size=len(groups))])
        diffs.append(f1_score(y[idx], pred_a[idx], labels=labels, average="macro", zero_division=0)
                     - f1_score(y[idx], pred_b[idx], labels=labels, average="macro", zero_division=0))
    return tuple(np.percentile(diffs, [2.5, 97.5]))


def verification_pairs(y, stores, contracts):
    """Pares (imagen, mueble declarado) por intercambio dentro del local (sección 5.9).

    Solo locales con 2+ convenios e imágenes de mueble con convenio en ese local.
    Devuelve listas paralelas: índice de imagen, índice declarado, es_correcto.
    """
    img, declared, correct = [], [], []
    for i, (label, s) in enumerate(zip(y, stores)):
        own = contracts.get(s, set())
        if len(own) < 2 or CLASSES[label] not in own:
            continue
        for c in own:
            img.append(i)
            declared.append(IDX[c])
            correct.append(IDX[c] == label)
    return np.array(img), np.array(declared), np.array(correct)


def verification(probs, y, stores, contracts):
    """Tasa de detección sobre pares discrepantes y de falsas alarmas sobre pares correctos."""
    img, declared, correct = verification_pairs(y, stores, contracts)
    pred = conditional_pred(probs, stores, contracts)[img]
    alert = pred != declared  # cualquier alerta: discrepancia o rechazo
    return {"detection_rate": float(alert[~correct].mean()), "false_alarm": float(alert[correct].mean()),
            "n_mismatch": int((~correct).sum()), "n_correct": int(correct.sum())}


def failed_criteria(m):
    """Criterios de la tabla 5.7 que `m` incumple. Lista vacía = configuración viable.
    `m` puede traer: f1_macro, recall_macro, reject_recall, false_alarm, quant_drop,
    lat_p50_ms, lat_p95_ms, mem_mb, size_mb. Las claves ausentes se reportan como pendientes."""
    higher = {"f1_macro", "recall_macro", "reject_recall"}
    return [k for k, t in THRESHOLDS.items()
            if k not in m or (m[k] < t if k in higher else m[k] > t)]

In [ ]:
import os, subprocess, time
from pathlib import Path
import pandas as pd
from IPython.display import clear_output

os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'  # cada proceso toma solo la memoria de GPU que usa
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
PARALLEL = 3
RUNS = [  # (modelo, experimento, sin máscara); los más lentos primero
    ('resnet50', 'A', False), ('resnet50', 'B', False),
    ('efficientnetb0', 'A', False), ('efficientnetb0', 'B', False),
    ('mobilenetv3', 'A', False), ('mobilenetv3', 'B', False),
    ('mobilenetv3', 'A', True),  # control del atajo de la superposición
]

def name_of(model, exp, nomask):
    return f"{exp}_{model}{'_nomask' if nomask else ''}"

def command(model, exp, nomask):
    name, saved = name_of(model, exp, nomask), f'{RUNS_DIR}/' + name_of(model, exp, nomask)
    if os.path.exists(f'{saved}/model.keras'):  # entrenado antes de una desconexión: se retoma
        steps = [f'mkdir -p runs && cp -r "{saved}" runs/']
    else:
        steps = [f'python -m tesis.train --model {model} --exp {exp}' + (' --no-mask' if nomask else ''),
                 f'cp -r runs/{name} "{RUNS_DIR}/"']
    steps += [f'python -m tesis.quantize runs/{name}',
              f'python -m tesis.evaluate runs/{name} model.keras',
              f'python -m tesis.evaluate runs/{name} model_int8.tflite',
              f'cp -r runs/{name} "{RUNS_DIR}/"']
    return ' && '.join(steps)

def status(name):
    h = Path(f'runs/{name}/history.csv')
    if Path(f'runs/{name}/model_int8.tflite').exists():
        return f'{name}: entrenado; evaluando'
    if not h.exists() or h.stat().st_size < 60:
        return f'{name}: época 1 en curso'
    d = pd.read_csv(h)
    return (f'{name}: {len(d)} épocas · val_accuracy última {d.val_accuracy.iloc[-1]:.3f}, '
            f'mejor {d.val_accuracy[d.val_loss.idxmin()]:.3f} · {len(d) - 1 - d.val_loss.idxmin()} sin mejorar')

Path('logs').mkdir(exist_ok=True)
pending = [r for r in RUNS if not os.path.exists(f'{RUNS_DIR}/{name_of(*r)}/metrics_model_int8.json')]
running, finished, t0 = {}, [], time.time()
while pending or running:
    while pending and len(running) < PARALLEL:
        r = pending.pop(0)
        running[name_of(*r)] = subprocess.Popen(command(*r), shell=True, stderr=subprocess.STDOUT,
                                                stdout=open(f'logs/{name_of(*r)}.log', 'w'))
    for name, p in list(running.items()):
        if p.poll() is not None:
            finished.append(f"{name}: {'listo, guardado en Drive' if p.returncode == 0 else 'FALLÓ, ver logs/' + name + '.log'}")
            del running[name]
    clear_output(wait=True)
    print(f'{(time.time() - t0) / 3600:.1f} h · en curso {len(running)} · en cola {len(pending)} · terminados {len(finished)}')
    for name in running:
        print('  ', status(name))
    for line in finished:
        print('  ', line)
    if running:
        time.sleep(60)
for line in finished:
    if 'FALLÓ' in line:
        name = line.split(':')[0]
        print(f'--- {name} (últimas líneas) ---'); print(''.join(open(f'logs/{name}.log').readlines()[-25:]))

## Prueba sin duplicados (sección 5.4)

92 fotos de la prueba están también, idénticas, en otro local: la misma foto archivada en dos carpetas de local (TIA 133
y TIA 546 comparten 16), copiada con otro nombre (`… (1).jpg`) o adjuntada de nuevo desde la galería en otra sumisión
de ODK. 88 tienen la copia en un local de entrenamiento o validación, así que son fuga: la deduplicación solo comparaba
fotos dentro de cada local. Desde aquí todas las métricas las excluyen. `tesis.build_dataset` ya las descarta al
reconstruir el conjunto y conserva la copia que no es de prueba, así que el entrenamiento no cambia.

In [ ]:
# Las 92 fotos, calculadas con tesis.split.cross_store_duplicates sobre data/hashes.csv (no viaja en el zip).
import glob, json
import numpy as np

DUPLICADOS = {
    '00ce0c8c_TimePhoto_20251117_120856.jpg', '042495e5_TimePhoto_20240715_110010.jpg',
    '053d60c6_TimePhoto_20250805_105145.jpg', '09293606_TimePhoto_20251117_121235.jpg',
    '09b0d69e_TimePhoto_20251117_121203.jpg', '0e25f7c3_TimePhoto_20251117_114833.jpg',
    '0ebc04a1_CC80306E-ACE3-4F70-A378-097ED2C5C398.jpg', '19e47d87_TimePhoto_20250814_150530.jpg',
    '1b703e65_TimePhoto_20250616_115535.jpg', '2182e8ad_TimePhoto_20250114_130137.jpg',
    '21ed02f9_c5ae4682-1a27-40ff-b751-5c3dd18e2b87.jpg', '22164c08_TimePhoto_20251117_121306.jpg',
    '24d0bad2_TimePhoto_20240312_103124.jpg', '287b1569_TimePhoto_20250915_104246.jpg',
    '2a3c992c_TimePhoto_20251226_110537.jpg', '2d8ebfb5_TimePhoto_20240104_085053.jpg',
    '2d9e63a8_TimePhoto_20251111_144047.jpg', '2ddbfbd1_TimePhoto_20250814_150445.jpg',
    '2ea57afe_TimePhoto_20250814_145445.jpg', '3102981c_TimePhoto_20250915_104201.jpg',
    '32ef2139_TimePhoto_20250114_130209.jpg', '3372533f_TimePhoto_20250814_152044.jpg',
    '35d809e6_TimePhoto_20250114_142335.jpg', '391bf01e_1791470143949.jpg',
    '3a0abc80_TimePhoto_20250916_160526.jpg', '3aa2bf64_c606af50-f922-41f3-a7fc-a20623c3f154.jpg',
    '3d82205e_TimePhoto_20240401_100047.jpg', '3f4fb107_TimePhoto_20250114_135108.jpg',
    '407511a6_TimePhoto_20251226_110740.jpg', '49f412f4_TC_00518.jpg', '4f4d1250_TimePhoto_20250702_161655.jpg',
    '4f5336a9_TimePhoto_20240214_131808.jpg', '53a15120_TimePhoto_20250805_105142.jpg',
    '53f3a053_TimePhoto_20240111_115057.jpg', '56c5b240_TimePhoto_20251117_120931.jpg',
    '5cde8764_TimePhoto_20240104_085254.jpg', '5e5311f8_TimePhoto_20250814_150954.jpg',
    '5f4b14c5_TimePhoto_20250114_141402.jpg', '689300c1_TimePhoto_20240618_143237.jpg',
    '68f58dd0_TimePhoto_20251111_144040.jpg', '69861d2f_TimePhoto_20241017_095414.jpg',
    '6a7b22f7_TimePhoto_20240104_085040.jpg', '6ee16264_TimePhoto_20250711_104343.jpg',
    '71f7e5a9_TimePhoto_20241008_085300.jpg', '72e28a0e_1792099512383.jpg', '74a533a7_TC_00516.jpg',
    '74fd697a_TimePhoto_20250915_104043.jpg', '7708c82d_TimePhoto_20250805_105151.jpg',
    '7b7bf131_TimePhoto_20251117_121254.jpg', '7b85fadc_TimePhoto_20240807_092345.jpg',
    '7cb18f20_TimePhoto_20240715_110110.jpg', '7d64376c_TimePhoto_20240715_105931.jpg',
    '7d70a33c_TimePhoto_20240105_150757.jpg', '7f3ff1e5_TimePhoto_20250120_123704.jpg',
    '8011fe99_TimePhoto_20251111_144036.jpg', '823d7ca6_TimePhoto_20241217_170124.jpg',
    '85ee64d4_TimePhoto_20240403_164903.jpg', '86c4b714_TimePhoto_20240401_094633.jpg',
    '87300a23_TimePhoto_20250114_140616.jpg', '88d8b406_TimePhoto_20250814_150317.jpg',
    '894e66a5_TimePhoto_20250915_104717.jpg', '8cce993d_TimePhoto_20240715_110120.jpg',
    '97743061_TimePhoto_20240912_123330.jpg', '98db6055_TimePhoto_20241115_164534.jpg',
    '99f2e897_TimePhoto_20250114_134848.jpg', '9a3af6aa_TimePhoto_20240715_105941.jpg',
    '9bfa6b19_TimePhoto_20250805_105208.jpg', 'a77d5195_TimePhoto_20250814_150354.jpg',
    'ab2eeb7d_TimePhoto_20240410_095946.jpg', 'acffffda_TimePhoto_20250822_112221.jpg',
    'aeaf3d14_TimePhoto_20240312_103048.jpg', 'af091914_TimePhoto_20251111_144033.jpg',
    'b5ed6345_20256997-a773-4f4b-94d0-d0178775c989.jpg', 'b604e5ca_TimePhoto_20240618_143137.jpg',
    'b63839ff_TimePhoto_20251111_144057.jpg', 'bc8eabdc_TimePhoto_20251226_110712.jpg',
    'c438f6c7_TimePhoto_20240618_143247.jpg', 'c4c6e284_TimePhoto_20250805_105148.jpg',
    'c738a863_TimePhoto_20250114_134709.jpg', 'cd29fcca_576ed733-537d-413a-a0f9-565a49b5a50c.jpg',
    'd207e800_TimePhoto_20251117_121113.jpg', 'd502421b_TimePhoto_20250320_162504.jpg',
    'd7b61a44_TimePhoto_20240715_110024.jpg', 'e131ec1a_TimePhoto_20251111_144050.jpg',
    'e3c1b324_TimePhoto_20251226_110637.jpg', 'e4eaee0c_TimePhoto_20251111_144100.jpg',
    'e7753744_TimePhoto_20250114_135235.jpg', 'e833e7ac_TimePhoto_20240618_143201.jpg',
    'f50c2b53_TimePhoto_20250114_134604.jpg', 'f817aa8e_8a250c1c-ca52-4740-ae9b-790e9ed08495.jpg',
    'ff0dd314_TimePhoto_20250814_152039.jpg', 'ff2dab3b_TimePhoto_20240401_094048.jpg'
}

def cargar(run, fmt='model'):
    """Prueba del run sin los duplicados y las probabilidades del formato, alineadas."""
    test = pd.read_csv(f'{RUNS_DIR}/{run}/test.csv')
    keep = ~test.path.str.rsplit('/', n=1).str[-1].isin(DUPLICADOS).values
    return test[keep].reset_index(drop=True), np.load(f'{RUNS_DIR}/{run}/probs_{fmt}.npy')[keep]

for f in sorted(glob.glob(f'{RUNS_DIR}/*/test.csv')):
    t = pd.read_csv(f)
    print(f'{f.split("/")[-2]}: {t.path.str.rsplit("/", n=1).str[-1].isin(DUPLICADOS).sum()} de {len(t):,} fotos de prueba fuera')

In [ ]:
# Resumen sobre la prueba sin duplicados. f1_presentes promedia solo las clases con ejemplos en prueba: la A tiene dos
# capturas inválidas reales y la B ninguna, y ese recall arrastra f1_macro y reject_recall.
from sklearn.metrics import f1_score
from tesis.config import CLASSES
from tesis.metrics import report

rows = []
for f in sorted(glob.glob(f'{RUNS_DIR}/*/probs_*.npy')):
    run, fmt = f.split('/')[-2], f.split('probs_')[-1][:-4]
    test, P = cargar(run, fmt)
    y = test.label.map(CLASSES.index).values
    m = report(y, P)
    rows.append({'run': run, 'formato': fmt, 'f1_macro': m['f1_macro'],
                 'f1_presentes': f1_score(y, P.argmax(1), labels=np.unique(y), average='macro'),
                 'recall_macro': m['recall_macro'], 'reject_recall': m['reject_recall'],
                 'size_mb': json.load(open(f'{RUNS_DIR}/{run}/metrics_{fmt}.json'))['size_mb']})
pd.DataFrame(rows).round(3)

## Experimento de cuantización: qué formato conserva la exactitud en el teléfono

La cuantización int8 completa, la estándar de TFLite, rompe las redes móviles ajustadas: en el experimento A,
EfficientNet-B0 pierde 8 puntos de F1-macro y MobileNetV3 30, mientras ResNet-50 no pierde nada pero pesa 23 MB. El
criterio de la tabla 5.7 es perder a lo sumo 2 puntos y pesar a lo sumo 20 MB. Aquí se comparan cuatro formatos
post-entrenamiento sobre los mismos modelos ya entrenados, sin reentrenar:

| Formato | Pesos | Activaciones | Calibración | Tamaño frente a float |
|---|---|---|---|---|
| `model_fp16` | float16 | float | no necesita | ½ |
| `model_dyn` | int8 | float (rango dinámico) | no necesita | ¼ |
| `model_int8` | int8 | int8 | 300 fotos de entrenamiento al azar | ¼ |
| `model_int8bal` | int8 | int8 | 100 fotos por clase | ¼ |

- **Hipótesis.** El daño está en las activaciones, no en los pesos: si es así, fp16 y dyn conservan la exactitud.
  int8bal prueba si la causa es la calibración: al azar, casi la mitad de las fotos son check outs y las clases
  chicas aportan pocas, así que sus activaciones pueden quedar fuera del rango calibrado.
- **Modelos.** Las tres arquitecturas del protocolo en los experimentos A y B: MobileNetV3 y EfficientNet-B0,
  candidatas al teléfono, y ResNet-50, la referencia, que no cabe en 20 MB en ningún formato (en int8 pesa 23 MB). El
  control sin máscara no se despliega.
- **Métrica (sección 5.7).** **Caída** = F1-macro de las 7 clases del modelo float32 − F1-macro del formato, en
  puntos, con IC 95 % por bootstrap pareado que remuestrea locales (2 000 réplicas). Como complemento, la caída sobre
  las clases con al menos 10 fotos en prueba: la prueba A tiene solo dos capturas inválidas reales, que mueven el
  F1-macro de 7 clases en 1/7 cada vez que se aciertan o se pierden. **Acuerdo** = fracción de fotos en que el formato
  predice lo mismo que el modelo float.
- **Criterios.** Los de la tabla 5.7 que se pueden medir aquí: F1-macro ≥ 0.85, recall-macro ≥ 0.90, recall de cada
  categoría de rechazo ≥ 0.90, caída ≤ 2 puntos y tamaño ≤ 20 MB. Latencia y memoria se miden en el Honor X6b; las falsas
  alarmas, en la verificación de la sección 5.9 (análisis del final).

TFLite evalúa en la CPU, foto por foto: es la parte lenta. Si Colab se desconecta, *Ejecutar todas* de nuevo: lo
terminado se salta.

Código nuevo respecto del zip: los formatos de cuantización.

In [ ]:
%%writefile tesis/quantize.py
"""Conversión a TFLite post-entrenamiento (sección 5.5, procedimiento estándar de TFLite).

Uso:
    python -m tesis.quantize runs/A_mobilenetv3              # model_fp32 y model_int8
    python -m tesis.quantize runs/A_mobilenetv3 --variantes  # model_fp16, model_dyn y model_int8bal
Formatos, de menos a más agresivo:
    fp32     sin cuantizar.
    fp16     pesos en float16; el teléfono calcula en float.
    dyn      pesos en int8, activaciones en float (rango dinámico): no necesita calibración.
    int8     pesos y activaciones en int8, calibrado con 300 imágenes de entrenamiento al azar.
    int8bal  igual, calibrado con 100 imágenes por clase: al azar, casi la mitad son check outs.
La calibración usa imágenes de entrenamiento sin aumentos.
"""
import sys
from pathlib import Path

import keras
import pandas as pd
import tensorflow as tf

from .data import dataset

N_CALIB = 300
N_CALIB_CLASE = 100


def saved_model(run):
    saved = run / "saved_model"
    keras.models.load_model(run / "model.keras").export(saved)
    return str(saved)


def converter(saved, optimize=True):
    conv = tf.lite.TFLiteConverter.from_saved_model(saved)
    if optimize:
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
    return conv


def int8(saved, calib):
    def representative():
        for x, _ in dataset(calib, batch=1):
            yield [x]

    conv = converter(saved)
    conv.representative_dataset = representative
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    # Entrada y salida quedan en float32: la app entrega RGB [0, 255] sin conocer la escala int8.
    return conv.convert()


def fp16(saved):
    conv = converter(saved)
    conv.target_spec.supported_types = [tf.float16]
    return conv.convert()


def convert(run, variantes=False):
    run = Path(run)
    saved = saved_model(run)
    train = pd.read_csv(run / "train.csv")
    if variantes:
        out = {"model_fp16": fp16(saved),
               "model_dyn": converter(saved).convert(),  # sin representative_dataset: rango dinámico
               "model_int8bal": int8(saved, train.sample(frac=1, random_state=0).groupby("label").head(N_CALIB_CLASE))}
    else:
        out = {"model_fp32": converter(saved, optimize=False).convert(),
               "model_int8": int8(saved, train.sample(N_CALIB, random_state=0))}
    for name, model in out.items():
        (run / f"{name}.tflite").write_bytes(model)
        print(f"{name}.tflite", f"{len(model) / 2**20:.1f} MB")


if __name__ == "__main__":
    convert(sys.argv[1], "--variantes" in sys.argv)

In [ ]:
# Cuantización: 3 formatos nuevos por run, los 6 runs en paralelo (TFLite evalúa en la CPU).
RUNS_CUANT = ['A_resnet50', 'B_resnet50', 'A_efficientnetb0', 'B_efficientnetb0', 'A_mobilenetv3', 'B_mobilenetv3']
FORMATOS = ['model.keras', 'model_int8.tflite', 'model_fp16.tflite', 'model_dyn.tflite', 'model_int8bal.tflite']

def falta(name, f):
    return not os.path.exists(f'{RUNS_DIR}/{name}/probs_{Path(f).stem}.npy')

def command_q(name):
    steps = [f'mkdir -p runs && cp -r "{RUNS_DIR}/{name}" runs/']
    if any(falta(name, f) for f in FORMATOS[2:]):
        steps.append(f'python -m tesis.quantize runs/{name} --variantes')
    steps += [f'python -m tesis.evaluate runs/{name} {f}' for f in FORMATOS if falta(name, f)]
    return ' && '.join(steps + [f'cp -r runs/{name} "{RUNS_DIR}/"'])

Path('logs').mkdir(exist_ok=True)
procs = {n: subprocess.Popen(command_q(n), shell=True, stderr=subprocess.STDOUT, stdout=open(f'logs/q_{n}.log', 'w'))
         for n in RUNS_CUANT if any(falta(n, f) for f in FORMATOS)}
t0 = time.time()
while any(p.poll() is None for p in procs.values()):
    clear_output(wait=True)
    print(f'{(time.time() - t0) / 60:.0f} min')
    for n, p in procs.items():
        hechos = len(list(Path(f'runs/{n}').glob('probs_*.npy')))
        print(f'   {n}: {hechos}/{len(FORMATOS)} formatos evaluados' + ('' if p.poll() is None else ' · terminado'))
    time.sleep(60)
for n, p in procs.items():
    print(n, 'listo, guardado en Drive' if p.returncode == 0 else
          'FALLÓ, últimas líneas:\n' + ''.join(open(f'logs/q_{n}.log').readlines()[-25:]))
if not procs:
    print('Todos los formatos ya estaban evaluados.')

In [ ]:
# Resultados (prueba sin duplicados): F1-macro, caída frente al modelo float32 con IC 95 % (bootstrap por local),
# acuerdo y criterios de la tabla 5.7 que se miden aquí. Tarda unos minutos.
from tesis.config import THRESHOLDS
from tesis.metrics import bootstrap_f1_diff, failed_criteria

TODAS = list(range(len(CLASSES)))
filas, por_clase = [], []
for n in RUNS_CUANT:
    test, _ = cargar(n)
    y = test.label.map(CLASSES.index).values
    clases = [i for i in TODAS if (y == i).sum() >= 10]
    f1 = lambda p, l=TODAS: f1_score(y, p, labels=l, average='macro', zero_division=0)
    probs = {Path(f).stem: cargar(n, Path(f).stem)[1] for f in FORMATOS}
    base = probs['model'].argmax(1)
    for fmt, P in probs.items():
        p = P.argmax(1)
        m = report(y, P) | {'size_mb': json.load(open(f'{RUNS_DIR}/{n}/metrics_{fmt}.json'))['size_mb'],
                            'quant_drop': f1(base) - f1(p)}
        fallas = failed_criteria(m)
        ic = '' if fmt == 'model' else '[{:.3f}, {:.3f}]'.format(*bootstrap_f1_diff(y, base, p, test.store.values))
        filas.append({'run': n, 'formato': fmt, 'MB': m['size_mb'], 'F1-macro': m['f1_macro'],
                      'recall-macro': m['recall_macro'], 'recall rechazo': m['reject_recall'],
                      'caída': m['quant_drop'], 'IC 95 % caída': ic, 'caída (≥10 fotos)': f1(base, clases) - f1(p, clases),
                      'acuerdo': (p == base).mean(), 'incumple': ', '.join(k for k in fallas if k in m)})
        por_clase.append({'run': n, 'formato': fmt, **dict(zip(CLASSES, f1_score(y, p, labels=TODAS, average=None,
                                                                                 zero_division=0)))})
cuant, por_clase = pd.DataFrame(filas), pd.DataFrame(por_clase)
print('Pendientes para todos: latencia y memoria (Honor X6b); las falsas alarmas, en el análisis del final.')
print('F1 por clase (experimento A): dónde pierde cada formato')
display(por_clase[por_clase.run.str.startswith('A')].set_index(['run', 'formato']).round(3))
cuant.round(3)

In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for a, exp in zip(axs, 'AB'):
    for n, d in cuant[cuant.run.str.startswith(exp)].groupby('run'):
        a.plot(d.MB, d['F1-macro'], 'o', label=n[2:])
        for _, r in d.iterrows():
            a.annotate(r.formato.replace('model_', '').replace('model', 'float'), (r.MB, r['F1-macro']),
                       xytext=(4, 2), textcoords='offset points', fontsize=8)
    a.axvline(THRESHOLDS['size_mb'], color='k', ls='--', lw=1, label='límite 20 MB')
    a.set_xscale('log'); a.set_title(f'Experimento {exp}'); a.set_xlabel('tamaño del archivo (MB, escala log)'); a.grid(alpha=.3)
axs[0].set_ylabel('F1-macro (7 clases)'); axs[0].legend()
plt.suptitle('Exactitud y tamaño por formato de cuantización'); plt.tight_layout(); plt.show()

## Análisis según la planificación: verificación, hipótesis 5.8 y errores por cadena

Sobre los runs del experimento A ya entrenados, sin reentrenar, y con la prueba sin duplicados:

- **Convenio de cada local.** El proyecto no tiene la maestra de convenios. Se aproxima con los muebles que aparecen en
  las fotos verificadas del local más los que ODK declara para él (el formulario solo despliega los contratados). Es un
  superconjunto del convenio, así que restringe la predicción menos que la maestra: las cifras que dependen de él son
  conservadoras.
- **Exactitud condicionada al local (sección 5.6).** La predicción se restringe a los muebles del convenio más las dos
  categorías de rechazo, como decide el sistema en operación.
- **Verificación (sección 5.9), dos evaluaciones.** *Pares por intercambio:* cada foto histórica de un mueble del
  convenio se declara como cada mueble del local. Declararla como otro mueble es una discrepancia que el sistema debe
  detectar; declararla como lo que es, un par correcto. Sobre estos pares se mide la tasa de falsas alarmas de la tabla
  5.7 (≤ 5 %). *Sumisiones reales de ODK:* cada foto con el mueble que el mercaderista declaró, incluidos los errores de
  campo, que la planificación daba por inexistentes y dejaba para trabajo futuro. En las dos, el sistema alerta cuando
  su predicción condicionada no es el mueble declarado, sea otro mueble o un rechazo.
- **Tabla 5.7.** Todos los criterios que se miden fuera del teléfono, ahora con las falsas alarmas. Latencia y memoria
  quedan pendientes del Honor X6b: sin ellas ninguna configuración puede declararse viable, ni se aplican la selección
  lexicográfica ni el frente de Pareto de F1-macro frente a latencia.
- **Capturas inválidas de campo.** La prueba tiene dos capturas inválidas reales: el recall de esa clase solo puede
  valer 0, 0.5 o 1, y arrastra el F1-macro, el recall-macro y el recall de rechazo. Como evidencia complementaria se
  predicen las 16 fotos oscuras de ODK de los locales de prueba (lente tapado, bolsillo, foto negra). No pasaron por la
  revisión manual, así que no entran a la prueba oficial y se reportan aparte.
- **Hipótesis 5.8.** ResNet-50 frente a la mejor arquitectura eficiente, con IC 95 % por bootstrap que remuestrea locales.
- **Objetivo 6.** Errores del candidato al teléfono por clase (matriz de confusión) y por cadena y fuente.

In [ ]:
# Convenios aproximados y las 16 fotos oscuras de ODK de los locales de prueba, predichas con cada formato (~2 min).
from tesis.config import REJECT
from tesis.evaluate import predict
from tesis.metrics import IDX, MUEBLES, conditional_pred, contracts, verification, verification_pairs

manifest = pd.read_csv('data/manifest.csv')
CONTRATOS = contracts(manifest)
locales = pd.read_csv('data/stores_split.csv')
NEGRAS = manifest[(manifest.source == 'odk') & (manifest.verified == 0) & (manifest.label == 'captura_invalida')
                  & manifest.store.isin(locales.store[locales.split == 'test'])]
RUNS_A = ['A_resnet50', 'A_efficientnetb0', 'A_mobilenetv3', 'A_mobilenetv3_nomask']
formatos = lambda n: [f for f in FORMATOS if os.path.exists(f'{RUNS_DIR}/{n}/probs_{Path(f).stem}.npy')]
P_NEGRAS = {(n, Path(f).stem): predict(Path(RUNS_DIR) / n / f, NEGRAS) for n in RUNS_A for f in formatos(n)}
print(f'{len(CONTRATOS)} locales con convenio aproximado · fotos oscuras de ODK en locales de prueba: {len(NEGRAS)}')

In [ ]:
# Tabla 5.7 sin el teléfono: criterios predictivos, falsas alarmas y detección (sección 5.9), exactitud condicionada,
# caída por cuantización y tamaño. Abajo, las 16 fotos oscuras de campo sumadas a la prueba.
filas, oscuras = [], []
for n in RUNS_A:
    test, _ = cargar(n)
    y, st = test.label.map(IDX).values, test.store.values
    hist = (test.source == 'hist').values
    odk = ((test.source == 'odk') & test.declarado.isin(MUEBLES)).values
    dec = test.declarado[odk].map(IDX).values
    ok = y[odk] == dec  # la foto muestra lo declarado
    for f in formatos(n):
        fmt = Path(f).stem
        P = cargar(n, fmt)[1]
        r = report(y, P, st, CONTRATOS)
        v = verification(P[hist], y[hist], st[hist], CONTRATOS)
        alerta = conditional_pred(P[odk], st[odk], CONTRATOS) != dec
        if fmt == 'model':  # el primero de FORMATOS
            f1_float = r['f1_macro']
        m = {'f1_macro': r['f1_macro'], 'recall_macro': r['recall_macro'], 'reject_recall': r['reject_recall'],
             'false_alarm': v['false_alarm'], 'quant_drop': f1_float - r['f1_macro'],
             'size_mb': json.load(open(f'{RUNS_DIR}/{n}/metrics_{fmt}.json'))['size_mb']}
        filas.append({'run': n, 'formato': fmt, 'MB': m['size_mb'], 'F1-macro': m['f1_macro'],
                      'recall-macro': m['recall_macro'], 'recall rechazo': m['reject_recall'],
                      'exactitud condicionada': r['conditional_accuracy'], 'falsas alarmas': v['false_alarm'],
                      'detección': v['detection_rate'], 'ODK: falsas alarmas': alerta[ok].mean(),
                      'ODK: errores detectados': alerta[~ok].mean(), 'caída': m['quant_drop'],
                      'incumple': ', '.join(k for k in failed_criteria(m) if k in m)})
        Pn = P_NEGRAS[n, fmt]
        ra = report(np.r_[y, np.full(len(Pn), IDX['captura_invalida'])], np.r_[P, Pn])
        oscuras.append({'run': n, 'formato': fmt, 'oscuras → captura inválida': (Pn.argmax(1) == IDX['captura_invalida']).sum(),
                        'oscuras → algún rechazo': np.isin(Pn.argmax(1), [IDX[c] for c in REJECT]).sum(),
                        'F1-macro': ra['f1_macro'], 'recall-macro': ra['recall_macro'], 'recall rechazo': ra['reject_recall']})
tabla = pd.DataFrame(filas)
print(f'Pares por intercambio (fotos históricas): {v["n_correct"]:,} correctos y {v["n_mismatch"]:,} discrepantes · '
      f'sumisiones reales de ODK: {ok.sum():,} correctas y {(~ok).sum()} errores de campo')
print('Pendientes para todos: latencia y memoria (Honor X6b).')
display(tabla.round(3))
print(f'Con las {len(NEGRAS)} fotos oscuras de campo sumadas a la prueba:')
pd.DataFrame(oscuras).round(3)

In [ ]:
# Verificación por par de clases (sección 5.9) del candidato al teléfono: el formato de ≤ 20 MB con mayor F1-macro.
# Fila: lo que muestra la foto; columna: lo declarado. Diagonal: falsas alarmas sobre los pares correctos; fuera de ella,
# detección de la discrepancia.
cand = tabla[(tabla.MB <= THRESHOLDS['size_mb']) & ~tabla.run.str.endswith('nomask')].sort_values('F1-macro').iloc[-1]
print(f'Candidato: {cand.run} {cand.formato}')
test, P = cargar(cand.run, cand.formato)
y, st = test.label.map(IDX).values, test.store.values
hist = (test.source == 'hist').values
img, dec, _ = verification_pairs(y[hist], st[hist], CONTRATOS)
pares = pd.DataFrame({'foto': np.array(CLASSES)[y[hist][img]], 'declarado': np.array(CLASSES)[dec],
                      'alerta': conditional_pred(P[hist], st[hist], CONTRATOS)[img] != dec})
display(pares.pivot_table('alerta', 'foto', 'declarado', aggfunc='mean').round(3))
print('Pares por celda:')
display(pares.pivot_table('alerta', 'foto', 'declarado', aggfunc='size').fillna(0).astype(int))
# Errores reales de campo en ODK: qué se declaró, qué muestra la foto y cuántos detecta el candidato.
odk = ((test.source == 'odk') & test.declarado.isin(MUEBLES)).values
campo = test[odk].assign(alerta=conditional_pred(P[odk], st[odk], CONTRATOS) != test.declarado[odk].map(IDX).values)
campo[campo.label != campo.declarado].groupby(['declarado', 'label']).alerta.agg(errores='size', detectados='sum')

In [ ]:
# Hipótesis 5.8: ¿ResNet-50 supera de forma significativa a la mejor arquitectura eficiente? IC 95 % de la diferencia
# de F1-macro (7 clases) por bootstrap pareado que remuestrea locales. Significativa si el IC excluye el cero.
f1a = tabla[tabla.formato == 'model'].set_index('run')['F1-macro']
mejor = f1a[['A_efficientnetb0', 'A_mobilenetv3']].idxmax()
test, P = cargar('A_resnet50')
y, pr = test.label.map(IDX).values, P.argmax(1)
f1 = lambda p: f1_score(y, p, labels=TODAS, average='macro', zero_division=0)
for run, fmt in dict.fromkeys([(mejor, 'model'), (cand.run, cand.formato)]):  # mejor eficiente y candidato
    pe = cargar(run, fmt)[1].argmax(1)
    lo, hi = bootstrap_f1_diff(y, pr, pe, test.store.values)
    print(f'ResNet-50 float32 − {run[2:]} {fmt}: {f1(pr) - f1(pe):+.3f}, IC 95 % [{lo:+.3f}, {hi:+.3f}] → '
          + ('diferencia significativa' if lo > 0 or hi < 0 else 'sin diferencia significativa'))
mb = tabla[tabla.run == 'A_resnet50'].MB.min()
print(f'ResNet-50 pesa {mb:.0f} MB en su formato más chico (límite {THRESHOLDS["size_mb"]} MB).')

In [ ]:
# Objetivo 6: errores del candidato por clase (precision, recall y matriz de confusión normalizada por fila, lo que
# muestra la foto) y por cadena y fuente, con la exactitud de las tres arquitecturas en float32 como referencia.
test, P = cargar(cand.run, cand.formato)
y = test.label.map(IDX).values
pred = np.array(CLASSES)[P.argmax(1)]
display(pd.DataFrame(report(y, P)['per_class']).T.round(3))
display(pd.crosstab(test.label, pred, rownames=['foto'], colnames=['predicción'], normalize='index').round(3))

aciertos = {}
for run, fmt in [(r, 'model') for r in RUNS_A[:3]] + [(cand.run, cand.formato)]:
    t, Pr = cargar(run, fmt)
    assert (t.path == test.path).all()
    aciertos[f'exactitud {run[2:]} {fmt}'] = Pr.argmax(1) == y
g = test.assign(cadena=test.store.str.split().str[0], pred=pred, **aciertos)
g['error'] = np.where(g.pred != g.label, g.label + ' → ' + g.pred, None)

def resumen(d):
    pres = d.label.value_counts()[lambda s: s >= 10].index
    errores = d.error.value_counts()
    return pd.Series({'fotos': len(d), 'locales': d.store.nunique(), **d[list(aciertos)].mean(),
                      'F1-macro candidato (clases ≥10 fotos)': f1_score(d.label, d.pred, labels=pres, average='macro', zero_division=0),
                      'error más frecuente': f'{errores.index[0]} ({errores.iloc[0]})' if len(errores) else ''})

g.groupby(['cadena', 'source']).apply(resumen, include_groups=False).infer_objects().round(3)